# Streaming responses from a local LLM using Ollama

This example demonstrates how to stream the response of a local LLM through the Ollama HTTP API from a Jupyter notebook. Instead of waiting for the complete answer, the notebook prints each fragment as the model produces it and reports the time to first token.

## 1. Setup

Make sure Ollama is installed, the local server is running, and the default model is available with `ollama pull gemma3:4b`.

In [ ]:
%pip install requests

In [ ]:
import json
import os
import time

import requests

OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma3:4b")

def query_model(prompt, model=OLLAMA_MODEL, temperature=0):
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": True,
        "options": {"temperature": temperature},
    }

    start = time.perf_counter()
    first_token = None
    chunks = []
    stats = {}
    with requests.post(f"{OLLAMA_HOST}/api/generate",
                       json=payload,
                       stream=True,
                       timeout=120) as response:
        response.raise_for_status()
        for line in response.iter_lines():
            if not line:
                continue
            chunk = json.loads(line)
            text = chunk.get("response", "")
            if text and first_token is None:
                first_token = time.perf_counter() - start
            print(text, end="", flush=True)
            chunks.append(text)
            if chunk.get("done"):
                stats = chunk
    latency = time.perf_counter() - start
    print()

    input_tokens = stats.get("prompt_eval_count", 0)
    output_tokens = stats.get("eval_count", 0)
    ttft = first_token if first_token is not None else latency
    print(f"\tModel: {stats.get('model', model)}")
    print(f"\tTime to first token: {ttft:.3f} seconds")
    print(f"\tLatency: {latency:.3f} seconds")
    print(f"\tInput tokens: {input_tokens}")
    print(f"\tOutput tokens: {output_tokens}")
    print(f"\tTotal tokens: {input_tokens + output_tokens}")

    return "".join(chunks).strip()

In [ ]:
prompt = "How many tokens are in your context window?"
print("User:", prompt)
print("Local LLM: ", end="", flush=True)
query_model(prompt)